In [1]:
!pip -q install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.1 MB/s eta 0:00:00


Question 4: Email Response Automation using Multiple Agents
   Task:
    Create a 3-agent flow using ConversableAgent and Groq:

ReaderAgent: Extracts main point from an email


ResponderAgent: Drafts a reply


FormatterAgent: Formats it as a proper email




In [5]:
from google.colab import userdata
from groq import Groq

# Read API key from Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)


class GroqLLM:
    def __init__(self, model="llama-3.3-70b-versatile"):
        self.model = model

    def generate(self, prompt):
        response = client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.7,
            max_tokens=1024,
        )

        return response.choices[0].message.content.strip()

Create LLM

In [6]:
llm = GroqLLM()

Install AutoGen

In [15]:
!pip uninstall -y autogen pyautogen autogen-agentchat
!pip install -q pyautogen==0.2.35

Found existing installation: pyautogen 0.2.35
Uninstalling pyautogen-0.2.35:
  Successfully uninstalled pyautogen-0.2.35


Import ConversableAgent

In [18]:
!pip install -q groq pyautogen==0.2.35

Repair the current runtime

In [22]:
import google.generativeai as genai

In [26]:
!pip install -q autogen-agentchat autogen-ext[openai]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 14.8 MB/s eta 0:00:00


Create ReaderAgent

In [28]:
class ReaderAgent:
    def __init__(self, llm):
        self.llm = llm

    def analyze(self, email):

        prompt = f"""
You are ReaderAgent.

Read the email and extract:

1. Main Purpose
2. Important Details
3. Requested Action

Email:
{email}
"""

        return self.llm.generate(prompt)

ResponderAgent

In [29]:
class ResponderAgent:
    def __init__(self, llm):
        self.llm = llm

    def reply(self, summary):

        prompt = f"""
You are ResponderAgent.

Write a professional email reply based on the following summary.

Summary:

{summary}
"""

        return self.llm.generate(prompt)

FormatterAgent

In [30]:
class FormatterAgent:
    def __init__(self, llm):
        self.llm = llm

    def format_email(self, draft):

        prompt = f"""
You are FormatterAgent.

Convert the following draft into a professional email.

Include:

Subject:
Greeting:
Body:
Closing:
Signature:

Draft:

{draft}
"""

        return self.llm.generate(prompt)

Create Agents

In [31]:
reader = ReaderAgent(llm)
responder = ResponderAgent(llm)
formatter = FormatterAgent(llm)

Sample Email

In [32]:
email = """
Hello Team,

Our client meeting scheduled for Friday has been postponed to next Monday at 3 PM.

Please update your calendars and prepare the revised presentation before Sunday evening.

Thanks,
John
"""

Run Workflow

In [33]:
summary = reader.analyze(email)

print("="*60)
print("ReaderAgent")
print("="*60)
print(summary)

reply = responder.reply(summary)

print("\n"+"="*60)
print("ResponderAgent")
print("="*60)
print(reply)

formatted = formatter.format_email(reply)

print("\n"+"="*60)
print("FormatterAgent")
print("="*60)
print(formatted)

ReaderAgent
Here are the extracted information:

1. **Main Purpose**: To inform the team about a change in the client meeting schedule.
2. **Important Details**: 
   - The client meeting has been postponed from Friday to next Monday.
   - The new meeting time is 3 PM.
   - The team needs to update their calendars.
   - A revised presentation is required.
   - The revised presentation should be prepared before Sunday evening.
3. **Requested Action**: 
   - Update calendars with the new meeting schedule.
   - Prepare the revised presentation before Sunday evening.

ResponderAgent
Subject: Update: Client Meeting Schedule and Revised Presentation

Dear Team,

I am writing to inform you that there has been a change in the client meeting schedule. The meeting, which was previously scheduled for this Friday, has been postponed to next Monday. The new meeting time is 3 PM.

In light of this change, please ensure that you update your calendars to reflect the revised meeting schedule. This will 

      Incoming Email
            │
            ▼
      ReaderAgent
            │
            ▼
      ResponderAgent
            │
            ▼
      FormatterAgent
            │
            ▼
      Professional Email